### CEO Lakebase (Autoscaling)

Provisions a **Lakebase Autoscaling** project (`w.postgres`) for the CEO Dashboard app,
creates the session tables, and writes `app.yaml` with the endpoint connection details.

The project creator is automatically granted superuser in the `databricks_postgres` database,
so DDL runs cleanly from this stage notebook.

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0" "psycopg[binary]>=3.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-ceo-sessions".lower())
ENDPOINT_PATH = f"projects/{PROJECT_ID}/branches/production/endpoints/primary"

In [ ]:
# packages installed in cell 1

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec
import time, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

# Create or reuse Autoscale project
is_new = False
try:
    project = w.postgres.get_project(name=f"projects/{PROJECT_ID}")
    print(f"\u267b\ufe0f Reusing existing Lakebase project: {PROJECT_ID}")
except Exception:
    is_new = True
    print(f"Creating Lakebase Autoscale project: {PROJECT_ID}")
    op = w.postgres.create_project(
        project=Project(spec=ProjectSpec(display_name=f"CEO Sessions ({CATALOG})", pg_version="17")),
        project_id=PROJECT_ID,
    )
    project = op.result()
    add(CATALOG, "postgres_projects", {"project_id": PROJECT_ID, "name": project.name})
    print(f"\u2705 Created project: {project.name}")

# For existing projects the endpoint is already RUNNING — only poll when freshly created
if is_new:
    print(f"Waiting for endpoint to be ready...")
    for attempt in range(40):
        try:
            ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
            state = str(ep.status.current_state) if ep.status else "UNKNOWN"
            if "RUNNING" in state or "AVAILABLE" in state or "ACTIVE" in state:
                print(f"\u2705 Endpoint ready (state={state})")
                break
            print(f"  [{attempt+1}/40] state={state}, retrying in 15s...")
        except Exception as e:
            print(f"  [{attempt+1}/40] not yet reachable: {e}, retrying in 15s...")
        time.sleep(15)
    else:
        raise TimeoutError(f"Endpoint {ENDPOINT_PATH} did not start within 10 minutes")
else:
    ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
    print(f"\u2705 Endpoint ready (state={ep.status.current_state})")

##### Connect and create session tables

In [ ]:
import psycopg

ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
host = ep.status.hosts.host
user = w.current_user.me().user_name

cred = w.postgres.generate_database_credential(endpoint=ENDPOINT_PATH)

conn_str = f"host={host} dbname=databricks_postgres user={user} password={cred.token} sslmode=require"

# Project creator has superuser — CREATE TABLE works without any GRANT
STATEMENTS = [
    "CREATE TABLE IF NOT EXISTS ceo_sessions (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), title TEXT NOT NULL DEFAULT 'New Session', created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW())",
    "CREATE TABLE IF NOT EXISTS ceo_messages (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), session_id UUID NOT NULL REFERENCES ceo_sessions(id) ON DELETE CASCADE, role TEXT NOT NULL CHECK (role IN ('user', 'assistant')), content TEXT NOT NULL, created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), documents_referenced JSONB DEFAULT '[]'::jsonb)",
    "CREATE INDEX IF NOT EXISTS idx_ceo_messages_session ON ceo_messages(session_id, created_at)",
    "CREATE INDEX IF NOT EXISTS idx_ceo_sessions_updated ON ceo_sessions(updated_at DESC)",
]

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        for stmt in STATEMENTS:
            cur.execute(stmt)
    conn.commit()

print(f"\u2705 Tables ready in {PROJECT_ID}/databricks_postgres")
print(f"   Host: {host}")

##### Write app.yaml for CEO Dashboard (injected at deploy time)

In [ ]:
import os, json

def _latest_from_uc_state(resource_type, key):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = '{resource_type}'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            val = json.loads(row.resource_data).get(key, "")
            if val:
                return val
    except Exception as e:
        print(f"⚠️ uc_state lookup failed for {resource_type}/{key}: {e}")
    return ""

# Supervisor endpoint + tile_id (tile_id drives the /ml/bricks/sa/<id> deep link)
supervisor_endpoint = _latest_from_uc_state("multi_agent_supervisors", "endpoint_name")
supervisor_tile_id  = _latest_from_uc_state("multi_agent_supervisors", "tile_id")

# Genie space IDs (match by title)
revenue_title = f"CEO Revenue & Orders Intelligence ({CATALOG})"
ops_title = f"CEO Operations Intelligence ({CATALOG})"
genie_revenue_id = ""
genie_ops_id = ""
try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'genie_spaces'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        if info.get("title") == revenue_title and not genie_revenue_id:
            genie_revenue_id = info.get("space_id", "")
        if info.get("title") == ops_title and not genie_ops_id:
            genie_ops_id = info.get("space_id", "")
except Exception as e:
    print(f"⚠️ Could not read genie IDs: {e}")

# KA tile IDs (match by name)
def _ka_tile_id(ka_name):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = 'knowledge_assistants'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = json.loads(row.resource_data)
            if info.get("name") == ka_name:
                return info.get("tile_id", "")
    except Exception as e:
        print(f"⚠️ Could not read KA tile {ka_name}: {e}")
    return ""

ka_inspection  = _ka_tile_id(f"{CATALOG}-inspection-knowledge")
ka_legal       = _ka_tile_id(f"{CATALOG}-ceo-legal")
ka_regulatory  = _ka_tile_id(f"{CATALOG}-ceo-regulatory")
ka_audits      = _ka_tile_id(f"{CATALOG}-ceo-audits")
ka_consultancy = _ka_tile_id(f"{CATALOG}-ceo-consultancy")

# SQL warehouse ID
warehouse_id = ""
try:
    WAREHOUSE_NAME = f"{CATALOG}-ceo-warehouse"
    existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
    if existing_wh:
        warehouse_id = existing_wh[0].id
except Exception as e:
    print(f"⚠️ Could not resolve warehouse: {e}")

app_yaml_path = os.path.abspath("../apps/ceo-dashboard/app.yaml")

app_yaml_contents = f"""command:
  - uvicorn
  - app.main:app
env:
  - name: LAKEBASE_ENDPOINT_PATH
    value: '{ENDPOINT_PATH}'
  - name: LAKEBASE_DATABASE_NAME
    value: 'databricks_postgres'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: CEO_SUPERVISOR_ENDPOINT
    value: '{supervisor_endpoint}'
  - name: CEO_SUPERVISOR_TILE_ID
    value: '{supervisor_tile_id}'
  - name: GENIE_ID_REVENUE
    value: '{genie_revenue_id}'
  - name: GENIE_ID_OPS
    value: '{genie_ops_id}'
  - name: KA_ID_INSPECTION
    value: '{ka_inspection}'
  - name: KA_ID_LEGAL
    value: '{ka_legal}'
  - name: KA_ID_REGULATORY
    value: '{ka_regulatory}'
  - name: KA_ID_AUDITS
    value: '{ka_audits}'
  - name: KA_ID_CONSULTANCY
    value: '{ka_consultancy}'
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse_id}'
"""

with open(app_yaml_path, "w") as f:
    f.write(app_yaml_contents)

print(f"\u2705 Wrote app.yaml to {app_yaml_path}")
print(f"   Supervisor:  {supervisor_endpoint}")
print(f"   Supervisor tile: {supervisor_tile_id}")
print(f"   Revenue Genie: {genie_revenue_id}")
print(f"   Ops Genie:     {genie_ops_id}")
print(f"   KA Inspection: {ka_inspection}")
print(f"   KA Legal:      {ka_legal}")
print(f"   KA Regulatory: {ka_regulatory}")
print(f"   KA Audits:     {ka_audits}")
print(f"   KA Consultancy:{ka_consultancy}")
print(f"   Warehouse:     {warehouse_id}")

In [ ]:
print(f"\u2705 CEO Lakebase stage complete")
print(f"   Project: {PROJECT_ID}")
print(f"   Endpoint: {ENDPOINT_PATH}")
print(f"   Host: {host}")